# Golden test — benchmark de modelos del detector de cabezas

Evalúa **cualquier** modelo entrenado contra el **golden test set congelado** (151 frames, 1241 cabezas).
Métrica que decide: **count-MAE** (error absoluto medio de conteo por frame) + sesgo (sobre/sub-cuenta) + desglose por cámara.
Réplica fiel de `data/golden/eval_golden.py`, en formato reutilizable.

> **Corre dentro del contenedor `mot-dev`** (JupyterLab con GPU + ultralytics). Rutas = `/workspace`.
> Autodescubre todos los `outputs/head_detector/*/weights/best.pt` + el base, así que cuando entrenes
> un modelo nuevo, esta evaluación lo incluye sola.

**Campeón actual = R5:** count-MAE **2.04** · recall **0.792** · mAP50 **0.872**.  
Config fija: `conf=0.25`, `iou(NMS)=0.5` (el validado — no subir el iou).

## 0. Configuración y descubrimiento de modelos

In [1]:
import os, glob
from pathlib import Path
import numpy as np
os.environ.setdefault('YOLO_CONFIG_DIR', '/tmp/Ultralytics')

REPO = Path('/workspace')
GOLD = REPO / 'data' / 'golden'
CONF, NMS, IMGSZ = 0.25, 0.5, 640

# --- autodescubrir modelos ---
MODELS = {}
base = REPO / 'models' / 'yolov5mu-head-base.pt'
if base.exists():
    MODELS['BASE'] = base
for w in sorted((REPO / 'outputs' / 'head_detector').glob('*/weights/best.pt')):
    name = w.parent.parent.name.replace('yolo-bus-head-', '').upper()
    MODELS[name] = w

# Para comparar solo algunos, descomenta y ajusta:
# SELECT = ['R5', 'MIN5']
SELECT = None
if SELECT:
    MODELS = {k: v for k, v in MODELS.items() if k in SELECT}

assert GOLD.exists(), f'no existe {GOLD}'
print('Golden:', GOLD)
print(f'conf={CONF}  iou(NMS)={NMS}\n')
print('Modelos a evaluar:')
for k, v in MODELS.items():
    print(f'  {k:8s} {v}')

Golden: /workspace/data/golden
conf=0.25  iou(NMS)=0.5

Modelos a evaluar:
  BASE     /workspace/models/yolov5mu-head-base.pt
  MIN5     /workspace/outputs/head_detector/yolo-bus-head-min5/weights/best.pt
  R1       /workspace/outputs/head_detector/yolo-bus-head-r1/weights/best.pt
  R2       /workspace/outputs/head_detector/yolo-bus-head-r2/weights/best.pt
  R3       /workspace/outputs/head_detector/yolo-bus-head-r3/weights/best.pt
  R4       /workspace/outputs/head_detector/yolo-bus-head-r4/weights/best.pt
  R5       /workspace/outputs/head_detector/yolo-bus-head-r5/weights/best.pt
  YOLO26M-CROWDHUMAN-HEAD4 /workspace/outputs/head_detector/yolo26m-crowdhuman-head4/weights/best.pt
  YOLO26S-CROWDHUMAN-HEAD /workspace/outputs/head_detector/yolo26s-crowdhuman-head/weights/best.pt


## 1. Cargar el ground-truth del golden

In [2]:
def gt_counts():
    d = {}
    for lf in glob.glob(f'{GOLD}/labels/val/*.txt'):
        d[os.path.basename(lf)[:-4]] = sum(1 for ln in open(lf) if len(ln.split()) == 5)
    return d

def cam_of(stem):                      # golden_<cam>_fXXXXXX
    return stem.split('_f')[0].replace('golden_', '')

gt   = gt_counts()
imgs = sorted(glob.glob(f'{GOLD}/images/val/*.jpg'))
print(f'{len(imgs)} imágenes · {sum(gt.values())} cabezas reales')
print('cámaras:', sorted({cam_of(os.path.basename(p)[:-4]) for p in imgs}))

151 imágenes · 1241 cabezas reales
cámaras: ['S08', 'v05', 'v16', 'video02']


## 2. Evaluar (P/R/mAP + count-MAE por modelo)

Para cada modelo: métricas de detección con `yolo val`, y el error de conteo prediciendo frame a frame.

In [3]:
import cv2
from ultralytics import YOLO

rows = []
for name, mp in MODELS.items():
    print(f'>>> {name} ...', flush=True)
    m = YOLO(str(mp))
    # --- detección ---
    met = m.val(data=str(GOLD / 'golden.yaml'), imgsz=IMGSZ, conf=CONF, iou=NMS,
                workers=2, verbose=False, plots=False)
    P, R, mAP50, mAP = met.box.mp, met.box.mr, met.box.map50, met.box.map
    # --- conteo ---
    abs_err, bias, per_cam = [], [], {}
    for p in imgs:
        stem = os.path.basename(p)[:-4]
        pred = len(m.predict(cv2.imread(p), conf=CONF, iou=NMS, verbose=False)[0].boxes)
        e = pred - gt.get(stem, 0)
        abs_err.append(abs(e)); bias.append(e)
        per_cam.setdefault(cam_of(stem), []).append(abs(e))
    rows.append({
        'modelo': name, 'P': P, 'R': R, 'mAP50': mAP50, 'mAP50-95': mAP,
        'MAE': float(np.mean(abs_err)), 'sesgo': float(np.mean(bias)),
        'por_camara': {c: round(float(np.mean(v)), 2) for c, v in sorted(per_cam.items())},
    })
print('\n✅ listo')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
>>> BASE ...
Ultralytics 8.4.0 🚀 Python-3.10.14 torch-2.2.2 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 15947MiB)
YOLOv5m summary (fused): 106 layers, 25,045,795 parameters, 0 gradients, 64.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 189.4±16.4 MB/s, size: 124.3 KB)
val: Scanning /workspace/data/golden/labels/val.cache... 151 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 151/151 57.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 6.4it/s 1.6s.2s
                   all        151       1241       0.85      0.314      0.593      0.322
Speed: 0.3ms preprocess, 8.2ms inference, 0.0ms loss, 0.2

## 3. Tabla de resultados (ordenada por count-MAE, menor = mejor)

In [4]:
import pandas as pd
df = (pd.DataFrame(rows)
        .sort_values('MAE')
        .reset_index(drop=True))
show = df[['modelo', 'MAE', 'sesgo', 'R', 'mAP50', 'mAP50-95', 'P']].copy()
for c in ['MAE', 'sesgo', 'R', 'mAP50', 'mAP50-95', 'P']:
    show[c] = show[c].map(lambda x: f'{x:.3f}' if c not in ('MAE', 'sesgo') else f'{x:+.2f}' if c == 'sesgo' else f'{x:.2f}')
print('GOLDEN TEST — conf 0.25 / iou 0.5  ·  campeón R5: MAE 2.04 / R 0.792\n')
print(show.to_string(index=False))

best = df.iloc[0]
print(f"\n🏆 mejor count-MAE: {best['modelo']} = {best['MAE']:.2f}  (sesgo {best['sesgo']:+.2f})")
if 'R5' in df['modelo'].values:
    r5 = df[df.modelo == 'R5'].iloc[0]['MAE']
    if best['modelo'] != 'R5':
        delta = r5 - best['MAE']
        verdict = 'MEJORA al campeón ✅ (considerar desplegar)' if delta > 0 else 'NO mejora ❌'
        print(f"   vs R5 ({r5:.2f}): {verdict}  Δ={delta:+.2f}")

GOLDEN TEST — conf 0.25 / iou 0.5  ·  campeón R5: MAE 2.04 / R 0.792

                  modelo  MAE sesgo     R mAP50 mAP50-95     P
                    MIN5 1.66 -1.37 0.809 0.873    0.634 0.877
                      R5 2.04 -1.87 0.792 0.872    0.632 0.901
                      R3 2.06 -1.89 0.758 0.854    0.654 0.894
                      R4 2.48 -2.40 0.758 0.862    0.713 0.925
                      R2 4.17 -4.11 0.513 0.690    0.424 0.864
                    BASE 5.13 -5.13 0.314 0.593    0.322 0.850
YOLO26M-CROWDHUMAN-HEAD4 5.25 -5.24 0.255 0.531    0.319 0.798
 YOLO26S-CROWDHUMAN-HEAD 5.66 -5.66 0.238 0.542    0.286 0.831
                      R1 6.85 -6.85 0.137 0.444    0.202 0.762

🏆 mejor count-MAE: MIN5 = 1.66  (sesgo -1.37)
   vs R5 (2.04): MEJORA al campeón ✅ (considerar desplegar)  Δ=+0.38


## 4. Desglose por cámara

`video02` es la cámara **no vista** en entrenamiento → el mejor termómetro de generalización.

In [5]:
cam_df = pd.DataFrame({r['modelo']: r['por_camara'] for r in rows}).T
cam_df = cam_df.reindex(df['modelo'].values)   # mismo orden que la tabla principal
print('MAE por cámara (menor = mejor):\n')
print(cam_df.to_string())

MAE por cámara (menor = mejor):

                           S08   v05   v16  video02
MIN5                      0.94  1.06  1.18     1.97
R5                        1.12  1.35  1.18     2.46
R3                        0.76  1.41  1.35     2.51
R4                        0.94  2.12  1.82     2.91
R2                        2.71  5.12  3.53     4.36
BASE                      4.24  5.06  5.00     5.31
YOLO26M-CROWDHUMAN-HEAD4  3.76  5.00  4.18     5.73
YOLO26S-CROWDHUMAN-HEAD   4.53  4.94  4.59     6.15
R1                        5.06  7.29  6.88     7.07
